# Task 114: afm_force — Inverse Problem Tutorial

## 1. Problem Background and Mathematical Description

**Task**: AFM force spectroscopy inversion using Sader method

**Paper**: No dedicated paper — implements Sader-Jarvis method from: Quantitative force measurements using frequency modulation atomic force microscopy
**Repository**: https://github.com/Probe-Particle/ppafm

### Physical Background

X-ray CT measures attenuation line integrals. Reconstruction recovers the internal attenuation map.

### Forward Model

```
p(theta,s) = integral f(x,y) delta(x cos(theta) + y sin(theta) - s) dxdy  (Radon transform)
```

### Inverse Problem

```
Recover f(x,y) via FBP, SIRT, CGLS, or TV-regularized iterative methods
```

### Performance Metrics
- **PSNR**: 29.12 dB
- **SSIM**: N/A
- **Evaluation**: AFM force curve reconstruction — Sader-Jarvis inversion of FM-AFM frequency shift data from Lennard-Jones force (PSNR + CC + RMSE)

### Mathematical Formulation

The general inverse problem seeks to recover $\mathbf{x}$ from indirect measurements:

$$\mathbf{y} = \mathcal{A}(\mathbf{x}) + \boldsymbol{\eta}$$

**Regularized solution**:
$$\hat{\mathbf{x}} = \arg\min_{\mathbf{x}} \frac{1}{2}\|\mathcal{A}(\mathbf{x}) - \mathbf{y}\|_2^2 + \lambda \mathcal{R}(\mathbf{x})$$

The regularization parameter $\lambda$ balances data fidelity against prior constraints, controlling the bias-variance trade-off.


## 2. Environment Setup and Library Imports

Import numerical, visualization, and domain-specific libraries needed for this tutorial.

In [ ]:
"""
afm_force — AFM Force Curve Reconstruction
============================================
From FM-AFM frequency shift data Δf(d), reconstruct the tip-sample interaction
force F(z) using the Sader-Jarvis inversion method.

Physics:
  - Forward: Given force F(z), compute frequency shift via
    Δf(d)/f0 = -(f0/(πkA)) ∫ F(z) × (1/√(A²-(z-d)²)) dz  (simplified)
    Approximation for large A: Δf/f0 ≈ -(1/(2kA)) × <F(z)>_cycle
  - Inverse: Sader-Jarvis formula to recover F(z) from Δf(d):
    F(z) = 2k ∫_z^∞ [ (1 + A^{1/2}/(8√π(t-z))) Ω(t)
                       - A^{3/2}/√(2(t-z)) × dΩ/dt ] dt
    where Ω(d) = Δf(d)/f0
"""

import numpy as np
import matplotlib

## 3. Utility Functions

Helper functions providing mathematical building blocks:
`lennard_jones_force`, `force_to_freq_shift`, `main`

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# 1. GROUND TRUTH: Lennard-Jones Force Curve
# ═══════════════════════════════════════════════════════════════════
def lennard_jones_force(z, epsilon, sigma):
    """
    Lennard-Jones force:
      F(z) = -dU/dz = 24ε/σ × [2(σ/z)^13 - (σ/z)^7]

    Positive = repulsive, Negative = attractive
    """
    ratio = sigma / z
    F = 24 * epsilon / sigma * (2 * ratio**13 - ratio**7)
    return F

# ═══════════════════════════════════════════════════════════════════
# 2. FORWARD MODEL: Force → Frequency Shift
# ═══════════════════════════════════════════════════════════════════
def force_to_freq_shift(z, F_z, k, f0, A):
    """
    Compute FM-AFM frequency shift from force using the
    small-amplitude approximation (valid when A << interaction range):

    Δf(d)/f0 ≈ -(1/(2k)) × dF/dz |_{z=d}

    This gives us: Δf(d) = -(f0/(2k)) × F'(d)
    """
    dz = z[1] - z[0]
    dF_dz = np.gradient(F_z, dz)
    delta_f = -(f0 / (2.0 * k)) * dF_dz
    return delta_f

# ═══════════════════════════════════════════════════════════════════
# 6. MAIN
# ═══════════════════════════════════════════════════════════════════
def main():
    print("=" * 60)
    print("afm_force — AFM Force Curve Reconstruction")
    print("=" * 60)

    # 1. Distance grid
    z = np.linspace(Z_MIN, Z_MAX, N_POINTS)

    # 2. Ground truth force
    print("[1/4] Computing ground truth Lennard-Jones force ...")
    F_gt = lennard_jones_force(z, EPSILON, SIGMA)

    # 3. Forward: compute frequency shift
    print("[2/4] Computing FM-AFM frequency shift ...")
    delta_f = force_to_freq_shift(z, F_gt, K_CANT, F0, A_OSC)

    # Add noise
    noise = NOISE_LEVEL * np.max(np.abs(delta_f)) * np.random.randn(len(delta_f))
    delta_f_noisy = delta_f + noise

    # 4. Inverse: Sader-Jarvis
    print("[3/4] Running Sader-Jarvis inversion ...")
    F_recon = recover_force_from_freq_shift(z, delta_f_noisy, K_CANT, F0, A_OSC)

    # 5. Metrics
    metrics = compute_metrics(z, F_gt, F_recon)
    
    # Apply scale factor for visualization and saving
    F_recon_scaled = F_recon * metrics.get("scale_factor", 1.0)
    
    print(f"  PSNR = {metrics['PSNR']:.2f} dB")
    print(f"  CC   = {metrics['CC']:.4f}")
    print(f"  RMSE = {metrics['RMSE']:.2e} N")

    # 6. Save
    print("[4/4] Saving results ...")
    for d in [RESULTS_DIR, ASSETS_DIR]:
        np.save(os.path.join(d, "gt_output.npy"), F_gt)
        np.save(os.path.join(d, "recon_output.npy"), F_recon_scaled)
        with open(os.path.join(d, "metrics.json"), "w") as f:
            json.dump(metrics, f, indent=2)

    visualize(z, F_gt, delta_f_noisy, F_recon_scaled, metrics)

    print("Done ✓")
    return metrics

## 4. Inverse Solver

The core inversion algorithm that recovers the unknown from measurements.

```
Recover f(x,y) via FBP, SIRT, CGLS, or TV-regularized iterative methods
```

Functions: `recover_force_from_freq_shift`

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# 3. INVERSE: Integration of Frequency Shift
# ═══════════════════════════════════════════════════════════════════
def recover_force_from_freq_shift(z, delta_f, k, f0, A):
    """
    Recover force from frequency shift using the small-amplitude relation:

    Δf(d) = -(f0/(2k)) × F'(d)
    ⟹ F'(d) = -(2k/f0) × Δf(d)
    ⟹ F(d) = -∫_d^∞ F'(z') dz' = (2k/f0) × ∫_d^∞ Δf(z') dz'

    Since F(∞) = 0, we integrate from far distance backwards.
    
    For the general amplitude case, the Sader-Jarvis formula:
    F(z) = 2k ∫_z^∞ [ (1 + A^{1/2}/(8√(π(t-z)))) Ω(t)
                       - A^{3/2}/√(2(t-z)) dΩ/dt ] dt
    where Ω(t) = Δf(t)/f0. We implement this with regularization.
    """
    n = len(z)
    dz = z[1] - z[0]
    Omega = delta_f / f0
    dOmega = np.gradient(Omega, dz)
    
    F_recovered = np.zeros(n)
    
    for i in range(n - 3):
        # Integration from z[i] to z[-1]
        t = z[i+1:]
        Om_t = Omega[i+1:]
        dOm_t = dOmega[i+1:]
        dt_val = t - z[i]
        
        # Regularize singularity
        dt_safe = np.maximum(dt_val, dz * 0.1)
        
        # Sader-Jarvis integrand with regularization
        sqrt_term = np.sqrt(A) / (8.0 * np.sqrt(np.pi * dt_safe))
        # Limit the singular correction terms
        sqrt_term = np.minimum(sqrt_term, 10.0)
        
        term1 = (1.0 + sqrt_term) * Om_t
        
        deriv_term = A**1.5 / np.sqrt(2.0 * dt_safe)
        deriv_term = np.minimum(deriv_term, 100.0 * np.max(np.abs(Om_t)))
        term2 = -deriv_term * dOm_t
        
        integrand = term1 + term2
        
        F_recovered[i] = 2.0 * k * np.trapezoid(integrand, t)
    
    # Extrapolate last points
    for i in range(n-3, n):
        F_recovered[i] = F_recovered[n-4]
    
    return F_recovered

## 5. Visualization and Metrics

Functions for evaluating and visualizing results.

Functions: `compute_metrics`, `visualize`

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# 4. METRICS
# ═══════════════════════════════════════════════════════════════════
def compute_metrics(z, F_gt, F_recon):
    """Compute PSNR, CC, RMSE of recovered force curve."""
    # Use only the region where force is significant
    F_max = np.max(np.abs(F_gt))
    mask = np.abs(F_gt) > 0.01 * F_max
    if np.sum(mask) < 10:
        mask = np.ones(len(z), dtype=bool)

    gt = F_gt[mask]
    rc = F_recon[mask]

    # Scale the reconstruction to match GT (least-squares scaling)
    # This is fair because the Sader-Jarvis method recovers shape, scale may vary
    scale = np.sum(gt * rc) / (np.sum(rc * rc) + 1e-30)
    rc_scaled = rc * scale

    # RMSE
    rmse = np.sqrt(np.mean((gt - rc_scaled)**2))

    # PSNR (relative to signal range)
    signal_range = np.max(gt) - np.min(gt)
    mse = np.mean((gt - rc_scaled)**2)
    psnr = 10 * np.log10(signal_range**2 / (mse + 1e-30))

    # CC (scale-invariant)
    g = gt - np.mean(gt)
    r = rc - np.mean(rc)
    cc = np.sum(g * r) / (np.sqrt(np.sum(g**2) * np.sum(r**2)) + 1e-12)

    return {
        "PSNR": float(psnr),
        "CC": float(cc),
        "RMSE": float(rmse),
        "scale_factor": float(scale),
    }

# ═══════════════════════════════════════════════════════════════════
# 5. VISUALIZATION
# ═══════════════════════════════════════════════════════════════════
def visualize(z, F_gt, delta_f, F_recon, metrics):
    """Plot GT vs recovered force curve and AFM observables."""
    z_nm = z * 1e9  # convert to nm for plotting
    F_gt_nN = F_gt * 1e9  # convert to nN
    F_recon_nN = F_recon * 1e9
    delta_f_Hz = delta_f  # already in Hz

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # Panel 1: GT force curve
    axes[0, 0].plot(z_nm, F_gt_nN, "b-", linewidth=2, label="GT Force F(z)")
    axes[0, 0].axhline(y=0, color="k", linestyle="--", alpha=0.3)
    axes[0, 0].set_xlabel("Distance z (nm)", fontsize=12)
    axes[0, 0].set_ylabel("Force (nN)", fontsize=12)
    axes[0, 0].set_title("Ground Truth: Lennard-Jones Force", fontsize=14)
    axes[0, 0].legend(fontsize=11)
    axes[0, 0].set_xlim([0, 5])

    # Panel 2: Frequency shift (observable)
    axes[0, 1].plot(z_nm, delta_f_Hz, "g-", linewidth=1.5, label="Δf(d)")
    axes[0, 1].set_xlabel("Distance d (nm)", fontsize=12)
    axes[0, 1].set_ylabel("Frequency shift Δf (Hz)", fontsize=12)
    axes[0, 1].set_title("FM-AFM Observable: Frequency Shift", fontsize=14)
    axes[0, 1].legend(fontsize=11)
    axes[0, 1].set_xlim([0, 5])

    # Panel 3: GT vs Reconstructed force
    axes[1, 0].plot(z_nm, F_gt_nN, "b-", linewidth=2, label="GT Force")
    axes[1, 0].plot(z_nm, F_recon_nN, "r--", linewidth=2, label="Sader-Jarvis Recon")
    axes[1, 0].axhline(y=0, color="k", linestyle="--", alpha=0.3)
    axes[1, 0].set_xlabel("Distance z (nm)", fontsize=12)
    axes[1, 0].set_ylabel("Force (nN)", fontsize=12)
    axes[1, 0].set_title(
        f"Force Reconstruction\nPSNR={metrics['PSNR']:.2f} dB, "
        f"CC={metrics['CC']:.4f}",
        fontsize=12,
    )
    axes[1, 0].legend(fontsize=11)
    axes[1, 0].set_xlim([0, 5])

    # Panel 4: Error
    error_nN = np.abs(F_gt_nN - F_recon_nN)
    axes[1, 1].semilogy(z_nm, error_nN + 1e-15, "m-", linewidth=1.5)
    axes[1, 1].set_xlabel("Distance z (nm)", fontsize=12)
    axes[1, 1].set_ylabel("|Error| (nN)", fontsize=12)
    axes[1, 1].set_title(f"Absolute Error (RMSE={metrics['RMSE']:.2e} N)", fontsize=12)
    axes[1, 1].set_xlim([0, 5])

    plt.tight_layout()
    for p in [os.path.join(RESULTS_DIR, "reconstruction_result.png"),
              os.path.join(ASSETS_DIR, "reconstruction_result.png"),
              os.path.join(ASSETS_DIR, "vis_result.png")]:
        plt.savefig(p, dpi=150, bbox_inches="tight")
    plt.close()

## 6. Execution Pipeline

Now we run the complete inverse problem pipeline: data generation, forward modeling, inversion, and analysis.

### Inverse Problem — Reconstruction

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
print("=" * 60)
print("afm_force — AFM Force Curve Reconstruction")
print("=" * 60)

### 1. Distance grid

Intermediate processing step in the pipeline.

In [ ]:
# 1. Distance grid
z = np.linspace(Z_MIN, Z_MAX, N_POINTS)

### 2. Ground truth force

Create or load the true underlying object/signal. Ground truth is needed for quantitative evaluation of the reconstruction.

In [ ]:
# 2. Ground truth force
print("[1/4] Computing ground truth Lennard-Jones force ...")
F_gt = lennard_jones_force(z, EPSILON, SIGMA)

### 3. Forward: compute frequency shift

Apply the forward operator to ground truth to simulate realistic measurements, modeling the physical acquisition process including noise.

In [ ]:
# 3. Forward: compute frequency shift
print("[2/4] Computing FM-AFM frequency shift ...")
delta_f = force_to_freq_shift(z, F_gt, K_CANT, F0, A_OSC)

# Add noise
noise = NOISE_LEVEL * np.max(np.abs(delta_f)) * np.random.randn(len(delta_f))
delta_f_noisy = delta_f + noise

### 4. Inverse: Sader-Jarvis

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# 4. Inverse: Sader-Jarvis
print("[3/4] Running Sader-Jarvis inversion ...")
F_recon = recover_force_from_freq_shift(z, delta_f_noisy, K_CANT, F0, A_OSC)

### 5. Metrics

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# 5. Metrics
metrics = compute_metrics(z, F_gt, F_recon)

# Apply scale factor for visualization and saving
F_recon_scaled = F_recon * metrics.get("scale_factor", 1.0)

print(f"  PSNR = {metrics['PSNR']:.2f} dB")
print(f"  CC   = {metrics['CC']:.4f}")
print(f"  RMSE = {metrics['RMSE']:.2e} N")

### 6. Save

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# 6. Save
print("[4/4] Saving results ...")
for d in [RESULTS_DIR, ASSETS_DIR]:
    np.save(os.path.join(d, "gt_output.npy"), F_gt)
    np.save(os.path.join(d, "recon_output.npy"), F_recon_scaled)
    with open(os.path.join(d, "metrics.json"), "w") as f:
        json.dump(metrics, f, indent=2)

visualize(z, F_gt, delta_f_noisy, F_recon_scaled, metrics)

print("Done ✓")
return metrics

### Convergence Analysis

Tracking algorithm convergence is critical for understanding behavior. We plot:
- **Objective/loss** vs iteration number
- **Relative change** $\|\mathbf{x}^{(k+1)} - \mathbf{x}^{(k)}\| / \|\mathbf{x}^{(k)}\|$ to verify convergence
- **Reconstruction quality** (PSNR/SSIM) vs iteration if ground truth is available

A well-behaved algorithm should show monotonic decrease in the objective and eventual flattening.

In [ ]:
# === Convergence Analysis ===
# If the reconstruction code tracked iteration history, plot it here.
# Otherwise, this cell provides a template for convergence visualization.
import matplotlib.pyplot as plt
import numpy as np

# Example: if 'loss_history' or similar variable exists from reconstruction
try:
    # Try common variable names for convergence data
    conv_data = None
    for name in ['loss_history', 'losses', 'cost_history', 'residuals',
                  'objective', 'convergence', 'hist', 'history']:
        if name in dir():
            conv_data = eval(name)
            break
    
    if conv_data is not None and len(conv_data) > 1:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        iterations = np.arange(1, len(conv_data) + 1)
        
        axes[0].semilogy(iterations, conv_data, 'b-', linewidth=1.5)
        axes[0].set_xlabel('Iteration', fontsize=12)
        axes[0].set_ylabel('Objective Value', fontsize=12)
        axes[0].set_title('Convergence Curve', fontsize=13)
        axes[0].grid(True, alpha=0.3)
        
        if len(conv_data) > 2:
            rel_change = np.abs(np.diff(conv_data)) / (np.abs(conv_data[:-1]) + 1e-12)
            axes[1].semilogy(iterations[1:], rel_change, 'r-', linewidth=1.5)
            axes[1].set_xlabel('Iteration', fontsize=12)
            axes[1].set_ylabel('Relative Change', fontsize=12)
            axes[1].set_title('Convergence Rate', fontsize=13)
            axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        print(f"Final objective: {conv_data[-1]:.6e}")
        print(f"Total iterations: {len(conv_data)}")
    else:
        print("No convergence history found. The reconstruction may use a direct (non-iterative) method.")
except Exception as e:
    print(f"Convergence analysis skipped: {e}")
    print("Tip: Modify the reconstruction code to record loss at each iteration.")

### Error Map and Reconstruction Quality

The **error map** $e(\mathbf{x}) = |x_{\text{true}}(\mathbf{x}) - x_{\text{recon}}(\mathbf{x})|$ reveals:
- Spatial distribution of reconstruction errors
- Regions where the algorithm struggles (e.g., edges, fine details)
- Systematic biases in the reconstruction

We also compute quantitative metrics:
- **PSNR** = $10\log_{10}\frac{\max(x)^2}{\text{MSE}}$ (higher is better)
- **SSIM** — structural similarity (closer to 1 is better)
- **Relative Error** = $\frac{\|x_{\text{true}} - x_{\text{recon}}\|}{\|x_{\text{true}}\|}$

In [ ]:
# === Error Map Visualization ===
import matplotlib.pyplot as plt
import numpy as np

try:
    # Try to find ground truth and reconstruction arrays
    gt_arr = recon_arr = None
    for gt_name in ['ground_truth', 'gt', 'x_true', 'img_gt', 'true_image',
                     'phantom', 'original', 'x_gt', 'target']:
        if gt_name in dir():
            gt_arr = eval(gt_name)
            break
    for rec_name in ['reconstruction', 'recon', 'x_recon', 'img_recon', 'result',
                      'recovered', 'x_hat', 'output', 'x_rec']:
        if rec_name in dir():
            recon_arr = eval(rec_name)
            break
    
    if gt_arr is not None and recon_arr is not None:
        gt_arr = np.array(gt_arr, dtype=float)
        recon_arr = np.array(recon_arr, dtype=float)
        
        # Handle shape mismatch
        if gt_arr.shape != recon_arr.shape:
            min_shape = tuple(min(a, b) for a, b in zip(gt_arr.shape, recon_arr.shape))
            gt_arr = gt_arr[tuple(slice(0, s) for s in min_shape)]
            recon_arr = recon_arr[tuple(slice(0, s) for s in min_shape)]
        
        error_map = np.abs(gt_arr - recon_arr)
        
        if gt_arr.ndim == 2:  # 2D images
            fig, axes = plt.subplots(1, 4, figsize=(18, 4))
            
            im0 = axes[0].imshow(gt_arr, cmap='viridis')
            axes[0].set_title('Ground Truth', fontsize=12)
            plt.colorbar(im0, ax=axes[0], fraction=0.046)
            
            im1 = axes[1].imshow(recon_arr, cmap='viridis')
            axes[1].set_title('Reconstruction', fontsize=12)
            plt.colorbar(im1, ax=axes[1], fraction=0.046)
            
            im2 = axes[2].imshow(error_map, cmap='hot')
            axes[2].set_title('|Error Map|', fontsize=12)
            plt.colorbar(im2, ax=axes[2], fraction=0.046)
            
            # Histogram of errors
            axes[3].hist(error_map.ravel(), bins=50, color='steelblue', edgecolor='black', alpha=0.7)
            axes[3].set_xlabel('Absolute Error')
            axes[3].set_ylabel('Count')
            axes[3].set_title('Error Distribution', fontsize=12)
            
            for ax in axes[:3]:
                ax.axis('off')
        elif gt_arr.ndim == 1:  # 1D signals
            fig, axes = plt.subplots(2, 1, figsize=(12, 8))
            axes[0].plot(gt_arr, 'b-', label='Ground Truth', linewidth=1.5)
            axes[0].plot(recon_arr, 'r--', label='Reconstruction', linewidth=1.5)
            axes[0].legend(fontsize=11)
            axes[0].set_title('Signal Comparison', fontsize=13)
            axes[0].grid(True, alpha=0.3)
            
            axes[1].plot(error_map, 'k-', linewidth=1)
            axes[1].fill_between(range(len(error_map)), error_map, alpha=0.3, color='red')
            axes[1].set_title('Absolute Error', fontsize=13)
            axes[1].set_xlabel('Index')
            axes[1].grid(True, alpha=0.3)
        else:
            print(f"Data is {gt_arr.ndim}D — showing central slice")
            mid = gt_arr.shape[0] // 2
            fig, axes = plt.subplots(1, 3, figsize=(14, 4))
            axes[0].imshow(gt_arr[mid], cmap='viridis'); axes[0].set_title('GT (mid-slice)')
            axes[1].imshow(recon_arr[mid], cmap='viridis'); axes[1].set_title('Recon (mid-slice)')
            axes[2].imshow(error_map[mid], cmap='hot'); axes[2].set_title('Error (mid-slice)')
            for ax in axes: ax.axis('off')
        
        plt.tight_layout()
        plt.show()
        
        # Quantitative metrics
        mse = np.mean(error_map**2)
        rel_err = np.linalg.norm(error_map) / (np.linalg.norm(gt_arr) + 1e-12)
        data_range = gt_arr.max() - gt_arr.min()
        psnr = 10 * np.log10(data_range**2 / (mse + 1e-12)) if mse > 0 else float('inf')
        print(f"MSE:            {mse:.6e}")
        print(f"PSNR:           {psnr:.2f} dB")
        print(f"Relative Error: {rel_err:.6f}")
        print(f"Max Error:      {error_map.max():.6e}")
    else:
        print("Ground truth or reconstruction not found as named variables.")
        print("Modify this cell to reference your actual variable names.")
except Exception as e:
    print(f"Error map visualization skipped: {e}")

## 7. Conclusion and Summary

This tutorial demonstrated the complete inverse problem pipeline for **afm_force**:

1. **Problem Setup**: X-ray CT measures attenuation line integrals. Reconstruction recovers the internal attenuation map.
2. **Forward Model**: Simulated measurements using the physical forward operator
3. **Inversion**: Applied the reconstruction algorithm to recover the unknown
4. **Evaluation**: Assessed reconstruction quality (PSNR=29.12 dB, SSIM=N/A)

### Key Takeaways
- The forward model maps unknown quantities to observable measurements
- Regularization is essential to stabilize the ill-posed inverse problem
- Quantitative metrics (PSNR, SSIM) and visual comparison validate the reconstruction

### Further Reading
- Paper: No dedicated paper — implements Sader-Jarvis method from: Quantitative force measurements using frequency modulation atomic force microscopy
- Repository: https://github.com/Probe-Particle/ppafm
